# Фаза 2 · распределение задач в Pyomo

In [ ]:
%pip -q install pyomo==6.11.0 highspy==1.16.0 openpyxl==3.1.5 XlsxWriter==3.2.9

In [ ]:
from pathlib import Path
from math import isfinite
from io import BytesIO
from google.colab import files
import pandas as pd
from IPython.display import display
import openpyxl
import pyomo.environ as p
from pyomo.opt import SolverStatus, TerminationCondition
import xlsxwriter


NAMES = ("Ольга Символокова", "Алина Недикова", "Никита Бредихин")
IDS = tuple(f"T{i}" for i in range(1, 19))
CAP = 17.5

## 1. Загрузка исходных оценок

In [ ]:
uploaded = files.upload()
if len(uploaded) != 2:
    raise ValueError('Выберите одновременно две исходные книги XLSX и повторите эту ячейку.')
books = [openpyxl.load_workbook(BytesIO(data), data_only=True) for data in uploaded.values()]
def select_book(required):
    matches = [book for book in books if required.issubset(book.sheetnames)]
    if len(matches) != 1:
        raise ValueError('Проверьте книги. Требуются листы: ' + ', '.join(sorted(required)))
    return matches[0]
effort = select_book({'Трудозатраты'})
pref = select_book({'Предпочтения', 'Распределение задач'})
if effort is pref:
    raise ValueError('Ожидаются две отдельные исходные книги.')
print('Выбраны файлы:', *uploaded.keys(), sep='\n')

In [ ]:
def read_scores(sheet, preferences=False):
    header = [cell.value for cell in sheet[5]]
    columns = [header.index(name) for name in NAMES]
    data = {}
    for row in list(sheet.values)[5:]:
        task, title = row[:2]
        if not any(v is not None for v in row):
            continue
        values = tuple(row[k] for k in columns)
        valid = all(type(v) in (int, float) and isfinite(v) and v > 0 for v in values)
        if preferences:
            valid = valid and all(int(v) == v and v <= 5 for v in values)
        if not valid or task in data or task not in IDS or not title:
            raise ValueError(f"Ошибка строки {sheet.title}: {task}, {values}")
        data[task] = (title, values)
    if set(data) != set(IDS):
        raise ValueError(f"Неполный набор задач на листе {sheet.title}")
    return data

def load_inputs(effort, pref):
    times = read_scores(effort["Трудозатраты"])
    scores = read_scores(pref["Предпочтения"], True)
    manual = {}
    for name, task, score in list(pref["Распределение задач"].values)[1:]:
        if task in manual or task not in IDS or name not in NAMES:
            raise ValueError(f"Ошибка ручного назначения {task}: {name}")
        j = NAMES.index(name)
        if score != scores[task][1][j] or times[task][0] != scores[task][0]:
            raise ValueError(f"Исходные листы расходятся для {task}")
        manual[task] = j
    if set(manual) != set(IDS):
        raise ValueError("Ручное распределение неполно")
    return times, scores, manual

times, scores, manual = load_inputs(effort, pref)

## 2. Оптимизация
Максимизируем сумму предпочтений. Каждой задаче назначается один участник, у каждого участника ровно шесть задач и нагрузка до 17,5 часа. Время задачи — среднее трёх оценок.

In [ ]:
def solve(times, scores):
    model = p.ConcreteModel()
    model.x = p.Var(IDS, range(3), domain=p.Binary)
    model.assign = p.Constraint(IDS, rule=lambda m,i: sum(m.x[i,j] for j in range(3)) == 1)
    model.count = p.Constraint(range(3), rule=lambda m,j: sum(m.x[i,j] for i in IDS) == 6)
    # Умножение среднего времени на три исключает округление исходных голосов.
    model.cap = p.Constraint(range(3), rule=lambda m,j:
        sum(sum(times[i][1])*m.x[i,j] for i in IDS) <= 3*CAP)
    model.obj = p.Objective(expr=sum(scores[i][1][j]*model.x[i,j]
        for i in IDS for j in range(3)), sense=p.maximize)
    result = p.SolverFactory("appsi_highs").solve(model)
    if (result.solver.status != SolverStatus.ok or
            result.solver.termination_condition != TerminationCondition.optimal):
        raise RuntimeError(str(result.solver))
    plan = {}
    for i in IDS:
        selected = [j for j in range(3) if p.value(model.x[i,j]) > 0.5]
        if len(selected) != 1:
            raise RuntimeError(f"Нарушение назначения {i}")
        plan[i] = selected[0]
    return plan, p.value(model.obj), str(result.solver.termination_condition)

def totals(times, scores, plan):
    result = []
    for j in range(3):
        tasks = [i for i in IDS if plan[i] == j]
        hours = sum(sum(times[i][1]) for i in tasks)/3
        points = sum(scores[i][1][j] for i in tasks)
        result.append((tasks, hours, points, len(tasks)==6 and hours<=CAP+1e-8))
    return result

best, value, status = solve(times, scores)
print("HiGHS:", status)

In [ ]:
rows = []
for label, plan in [('Принято', manual), ('Оптимум', best)]:
    group = totals(times, scores, plan)
    for person, (tasks, hours, points, feasible) in zip(NAMES, group):
        minutes = round(hours * 60)
        rows.append([label, person, ', '.join(tasks), len(tasks), f'{minutes//60} ч {minutes%60} мин', points, feasible])
    print(label, '—', sum(row[2] for row in group), 'баллов')
optimal = totals(times, scores, best)
assert all(row[3] for row in optimal) and abs(sum(row[2] for row in optimal) - value) < 1e-8
display(pd.DataFrame(rows, columns=['Вариант','Участник','Задачи','Количество','Нагрузка','Баллы','Допустимо']))

## 3. Скачать результат
Книга содержит исходные баллы, назначения и сводку двух вариантов.

In [ ]:
def write_book(path, times, scores, manual, best, status):
    with xlsxwriter.Workbook(path) as w:
        s = w.add_worksheet("Назначения")
        head = w.add_format({"bold":True,"bg_color":"#21475B","font_color":"white","text_wrap":True,"valign":"vcenter"})
        hour = w.add_format({"num_format":"0.00"})
        s.write_row(0,0,["Код","Учебная задача"]+[f"Часы · {n.split()[0]}" for n in NAMES]+
                    [f"Баллы · {n.split()[0]}" for n in NAMES]+["Среднее, ч","Принято","Оптимум","Балл плана","Балл оптимума"],head)
        s.set_column("A:A",9); s.set_column("B:B",78); s.set_column("C:I",14); s.set_column("J:K",23); s.set_column("L:M",15)
        s.set_row(0,38); s.freeze_panes(1,2); s.autofilter("A1:M19")
        for r,i in enumerate(IDS,1):
            a=r+1
            s.write_row(r,0,[i,times[i][0],*times[i][1],*scores[i][1]])
            s.write_formula(r,8,f"=AVERAGE(C{a}:E{a})",hour,sum(times[i][1])/3)
            for k,plan in enumerate([manual,best]):
                j=plan[i]; s.write(r,9+k,NAMES[j])
                col="JK"[k]
                s.write_formula(r,11+k,f"=INDEX(F{a}:H{a},1,MATCH({col}{a},Сводка!$A$2:$A$4,0))",None,scores[i][1][j])
        s.data_validation("C2:E19",{"validate":"decimal","criteria":">","value":0})
        s.data_validation("F2:H19",{"validate":"integer","criteria":"between","minimum":1,"maximum":5})
        s.data_validation("J2:K19",{"validate":"list","source":list(NAMES)})
        q=w.add_worksheet("Сводка");q.set_column("A:A",27);q.set_column("B:I",16);q.set_row(0,38)
        q.write_row(0,0,["Участник","План, задач","План, ч","План, балл","Допустим","Опт., задач","Опт., ч","Опт., балл","Допустим"],head)
        for k,plan in enumerate([manual,best]):
            group=totals(times,scores,plan)
            col="JK"[k]; val="LM"[k]; start=1+4*k
            for r,(tasks,h,b,ok) in enumerate(group,1):
                a=r+1;q.write(r,0,NAMES[r-1])
                rng=f"Назначения!{col}$2:{col}$19"
                for offset,(formula,cached) in enumerate([(f"COUNTIF({rng},A{a})",len(tasks)),
                    (f"SUMIF({rng},A{a},Назначения!I$2:I$19)",h),
                    (f"SUMIF({rng},A{a},Назначения!{val}$2:{val}$19)",b),
                    (f"AND({'BF'[k]}{a}=6,{'CG'[k]}{a}<=$B$7)",ok)]):
                    q.write_formula(r,start+offset,"="+formula,hour if offset==1 else None,cached)
            for off,letter,value in [(0,"BF"[k],18),(1,"CG"[k],sum(x[1] for x in group)),(2,"DH"[k],sum(x[2] for x in group))]:
                q.write_formula(4,start+off,f"=SUM({letter}2:{letter}4)",hour if off==1 else None,value)
        q.write("A5","Итого");q.write_row("A7",["Предел, ч",CAP]);q.write_row("A8",["Pyomo / HiGHS",status])
        q.write("A10","После изменения оценок запустите Python заново, чтобы обновить оптимум.")

output = "Фаза_2_Оптимизация.xlsx"
write_book(output, times, scores, manual, best, status)
files.download(output)